# GameRec: Models Demo (Popularity Baseline & Content-Based Filtering)
This notebook demonstrates the first two recommendation models built for Steam games:
1. **Model A: Bayesian Popularity Baseline** (with empirical shrinkage & genre slicing)
2. **Model B: Sparse Content-Based Filtering** (multi-modal TF-IDF on tags, descriptions, and titles)

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from gamerec.popularity_model import PopularityRecommender
from gamerec.content_model import ContentBasedRecommender

# Load persisted models
models_dir = project_root / "models"
pop_model = PopularityRecommender.load(models_dir / "popularity_model.joblib")
content_model = ContentBasedRecommender.load(models_dir / "content_model.joblib")
print("Models successfully loaded from artifacts!")

## 1. Popularity Baseline Recommendations
### Overall Top Popular Games

In [2]:
pop_overall = pop_model.recommend(k=5, ranking_metric="weighted_popularity")
for r in pop_overall:
    print(f"#{r['rank']}: {r['title']} (Score: {r['score']:.2f}, {r['positive_ratio']}% on {r['user_reviews']:,} reviews)")

### Tag-Specific Popular Games (e.g., Roguelike, RPG, Cyberpunk)

In [3]:
rogue_pop = pop_model.recommend(k=5, tag="Roguelike", min_reviews=500)
for r in rogue_pop:
    print(f"#{r['rank']}: {r['title']} ({r['positive_ratio']}% positive, {r['user_reviews']:,} reviews)")
    print(f"    {r['explanation']}")

## 2. Content-Based Recommendations: Single Seed Game
Test with iconic titles: Half-Life 2 (App ID 220), Tomb Raider (App ID 203160), Mount & Blade (App ID 48700)

In [4]:
hl2_recs = content_model.recommend_similar(app_id=220, k=5)
print("Recommendations for Half-Life 2:")
for r in hl2_recs:
    print(f"  {r['rank']}. {r['title']} (Similarity: {r['similarity_score']:.3f})")
    print(f"     Shared Tags: {r['shared_tags']}")
    print(f"     {r['explanation']}")

## 3. Content-Based Recommendations: Multi-Item User Taste Profile
Construct a composite taste profile vector from multiple games with user engagement weights.

In [5]:
user_profile = [
    {"app_id": 48700, "weight": 2.0},  # Mount & Blade: Warband (High weight)
    {"app_id": 203160, "weight": 1.0}, # Tomb Raider
]
profile_recs = content_model.recommend_user_profile(user_profile, k=5, min_reviews=100)
print("User Taste Profile Recommendations:")
for r in profile_recs:
    print(f"  {r['rank']}. {r['title']} (Profile Sim: {r['similarity_score']:.3f})")
    print(f"     Overlapping Tags: {r['shared_tags']}")
    print(f"     {r['explanation']}")